In [ ]:
!curl -o https://ranasinghiitkgp.medium.com/feature-engineering-using-featuretools-with-code-10f8c83e5f68

In [1]:
import featuretools as ft
import numpy as np
import pandas as pd

In [3]:
train = pd.read_csv("autoFETools.csv")


In [4]:
train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [7]:
train.isna().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [8]:
train['Item_Weight'].fillna(train['Item_Weight'].mean(), inplace = True)
train['Outlet_Size'].fillna("missing", inplace = True)

C:\Users\H P\AppData\Local\Temp\ipykernel_39196\2089450865.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['Item_Weight'].fillna(train['Item_Weight'].mean(), inplace = True)
C:\Users\H P\AppData\Local\Temp\ipykernel_39196\2089450865.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves

In [9]:
fat_content_dict = {'Low Fat':0, 'Regular':1, 'LF':0, 'reg':1, 'low fat':0}
train['Item_Fat_Content'] = train['Item_Fat_Content'].replace(fat_content_dict, regex=True)

train['id'] = train['Item_Identifier'] + train['Outlet_Identifier']
train.drop(['Item_Identifier'], axis=1, inplace=True)

C:\Users\H P\AppData\Local\Temp\ipykernel_39196\543229137.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Item_Fat_Content'] = train['Item_Fat_Content'].replace(fat_content_dict, regex=True)


In [ ]:
test_Item_Identifier = test['Item_Identifier']
test_Outlet_Identifier = test['Outlet_Identifier']
sales = train['Item_Outlet_Sales']
train.drop(['Item_Outlet_Sales'], axis=1, inplace=True)

In [10]:
train.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,id
0,9.30,0,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,FDA15OUT049
1,5.92,1,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,DRC01OUT018
2,17.50,0,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,FDN15OUT049
3,19.20,1,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,missing,Tier 3,Grocery Store,732.3800,FDX07OUT010
4,8.93,0,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,NCD19OUT013


In [15]:
from sklearn.preprocessing import LabelEncoder
colonnes_a_encoder = ['Item_Type', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type','Outlet_Identifier']
le = LabelEncoder()
for col in colonnes_a_encoder:
    train[col] = le.fit_transform(train[col].astype(str)) 

train.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,id
0,9.30,0,0.016047,2,249.8092,9,1999,1,0,1,3735.1380,FDA15OUT049
1,5.92,1,0.019278,12,48.2692,3,2009,1,2,2,443.4228,DRC01OUT018
2,17.50,0,0.016760,8,141.6180,9,1999,1,0,1,2097.2700,FDN15OUT049
3,19.20,1,0.000000,4,182.0950,0,1998,3,2,0,732.3800,FDX07OUT010
4,8.93,0,0.000000,7,53.8614,1,1987,0,2,1,994.7052,NCD19OUT013


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

In [17]:
features = ['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP',
            'Outlet_Identifier', 'Outlet_Establishment_Year', 'Outlet_Size',
            'Outlet_Location_Type', 'Outlet_Type']

X = train[features]
y = train['Item_Outlet_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [18]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [19]:
y_pred = model.predict(X_test)

print("MSE :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R² score :", r2_score(y_test, y_pred))

MSE : 1142.2525352352604
R² score : 0.5199576072316732


Application autofeat

In [38]:
from autofeat import AutoFeatRegressor 

af = AutoFeatRegressor( feateng_steps=2,n_jobs=-1)  
X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)
X_train_af.head()
print(f"Nombre de nouvelles features créées : {X_train_af.shape[1] - X_train.shape[1]}")

model_af = LinearRegression()
model_af.fit(X_train_af, y_train)
y_pred_af = model_af.predict(X_test_af)

mse = mean_squared_error(y_test, y_pred_af)
r2 = r2_score(y_test, y_pred_af)
print("\n Performance APRÈS AutoFeat")
print(f"Erreur quadratique moyenne (MSE) : {np.sqrt(mse):.2f}")
print(f"Score R² : {r2:.2f}")

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:


Nombre de nouvelles features créées : 26

 Performance APRÈS AutoFeat
Erreur quadratique moyenne (MSE) : 1021.68
Score R² : 0.62


c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Application featuretools

In [22]:

# creating and entity set 'es'
es = ft.EntitySet(id = 'sales')

# adding a dataframe 
es.add_dataframe(dataframe_name = 'bigmart', dataframe = train, index = 'id')

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(


Entityset: sales
  DataFrames:
    bigmart [Rows: 8523, Columns: 12]
  Relationships:
    No relationships

In [24]:
es.normalize_dataframe(base_dataframe_name='bigmart', new_dataframe_name='outlet', index = 'Outlet_Identifier', 
            additional_columns = ['Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type'])


Entityset: sales
  DataFrames:
    bigmart [Rows: 8523, Columns: 8]
    outlet [Rows: 10, Columns: 5]
  Relationships:
    bigmart.Outlet_Identifier -> outlet.Outlet_Identifier

In [27]:
feature_matrix, feature_names = ft.dfs(entityset=es, 
target_dataframe_name = 'bigmart', 
max_depth = 2, 
verbose = 1, 
n_jobs = 1)

Built 48 features
Elapsed: 00:00 | Progress:  48%|████▊     

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function max at 0x000002A4AD2857E0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  ).agg(to_agg)
c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function min at 0x000002A4AD285900> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  ).agg(to_agg)
c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function sum at 0x000002A4AD2851B0> is currently using SeriesGro

Elapsed: 00:00 | Progress: 100%|██████████


In [28]:
feature_matrix.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Item_Outlet_Sales,outlet.Outlet_Establishment_Year,outlet.Outlet_Size,outlet.Outlet_Location_Type,...,outlet.STD(bigmart.Item_Outlet_Sales),outlet.STD(bigmart.Item_Type),outlet.STD(bigmart.Item_Visibility),outlet.STD(bigmart.Item_Weight),outlet.SUM(bigmart.Item_Fat_Content),outlet.SUM(bigmart.Item_MRP),outlet.SUM(bigmart.Item_Outlet_Sales),outlet.SUM(bigmart.Item_Type),outlet.SUM(bigmart.Item_Visibility),outlet.SUM(bigmart.Item_Weight)
id,,,,,,,,,,,,,,,,,,,,,
FDA15OUT049,9.30,0,0.016047,2,249.8092,9,3735.1380,1999,1,0,...,1513.289464,4.400421,0.044602,4.617003,334.0,130476.8598,2.183970e+06,6227.0,56.549156,12013.225
DRC01OUT018,5.92,1,0.019278,12,48.2692,3,443.4228,2009,1,2,...,1375.932889,4.474619,0.045386,4.689009,330.0,131477.7724,1.851823e+06,6293.0,56.621454,11946.465
FDN15OUT049,17.50,0,0.016760,8,141.6180,9,2097.2700,1999,1,0,...,1513.289464,4.400421,0.044602,4.617003,334.0,130476.8598,2.183970e+06,6227.0,56.549156,12013.225
FDX07OUT010,19.20,1,0.000000,4,182.0950,0,732.3800,1998,3,2,...,271.014855,4.304091,0.072047,4.638683,196.0,78131.5646,1.883402e+05,3639.0,56.308832,7166.800
NCD19OUT013,8.93,0,0.000000,7,53.8614,1,994.7052,1987,0,2,...,1533.531664,4.404479,0.044235,4.666798,326.0,131809.0156,2.142664e+06,6120.0,55.879859,12121.730


In [32]:
X_ft = feature_matrix.drop(columns=['Item_Outlet_Sales'])
y_ft = feature_matrix['Item_Outlet_Sales']

In [33]:
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_ft, y_ft, test_size=0.20, random_state=42)

In [34]:
model = LinearRegression()
model.fit(X_train_fe, y_train_fe)

LinearRegression()

In [37]:
y_pred_fe = model.predict(X_test_fe)

print("MSE :", np.sqrt(mean_squared_error(y_test_fe, y_pred_fe)))
print("R² score :", r2_score(y_test_fe, y_pred_fe))

MSE : 1067.4681808342873
R² score : 0.5807575914505649
